In [1]:
"""
EXPERIMENT 6: TD(0) Prediction & Q-Learning
Key Idea: Learn after EVERY step. Uses bootstrapping (next state's value).
TD Error (delta) = R + gamma*V(s') - V(s)  <-- how wrong was our estimate?
"""
import numpy as np
from collections import defaultdict

# --- Same Grid World ---
class GridWorld:
    def reset(self): self.s = (0,0); return self.s
    def step(self, a):
        r,c = self.s
        if a==0: r=max(r-1,0)
        elif a==1: r=min(r+1,3)
        elif a==2: c=max(c-1,0)
        else: c=min(c+1,3)
        self.s = (r,c)
        if self.s==(3,3): return self.s,+10,True
        if self.s in [(1,1),(2,2)]: return self.s,-10,True
        return self.s,-1,False

env = GridWorld()

# ── PART A: TD(0) Prediction ───────────────────────────
# Update V(s) after every step using: V(s) += a[R + g*V(s') - V(s)]
V = defaultdict(float)

for _ in range(500):
    s, done = env.reset(), False
    while not done:
        a = np.random.randint(4)                  # random policy
        ns, r, done = env.step(a)
        # TD Update: nudge V(s) toward the TD target
        td_error = r + 0.9 * V[ns] * (not done) - V[s]
        V[s] += 0.1 * td_error                   # alpha=0.1
        s = ns

print("── TD(0) Prediction: V(s) ──")
for r in range(4): print([round(V[(r,c)],1) for c in range(4)])

# ── PART B: Q-Learning ────────────────────────────────
# Off-policy: explores with e-greedy, but updates using BEST next action
# Q(s,a) += a[R + g*max(Q(s',a')) - Q(s,a)]
Q = defaultdict(lambda: np.zeros(4))

for _ in range(2000):
    s, done = env.reset(), False
    while not done:
        a = np.random.randint(4) if np.random.rand()<0.1 else np.argmax(Q[s])
        ns, r, done = env.step(a)
        # Use max over next actions (greedy target) — that's what makes it Q-learning
        td_error = r + 0.9 * np.max(Q[ns]) * (not done) - Q[s][a]
        Q[s][a] += 0.1 * td_error
        s = ns

arrows = ['^','v','<','>']
print("\n── Q-Learning: Optimal Policy ──")
for r in range(4): print([arrows[np.argmax(Q[(r,c)])] for c in range(4)])

# Derive V(s) from Q: best action's value
print("\n── Q-Learning: V(s) = max_a Q(s,a) ──")
for r in range(4): print([round(max(Q[(r,c)]),1) for c in range(4)])

── TD(0) Prediction: V(s) ──
[-9.8, -9.8, -8.7, -6.7]
[-9.8, 0.0, -8.9, -4.9]
[-9.3, -9.2, 0.0, 1.5]
[-8.5, -7.7, -2.9, 0.0]

── Q-Learning: Optimal Policy ──
['>', '>', '>', 'v']
['v', '^', '>', 'v']
['>', 'v', '^', 'v']
['>', '>', '>', '^']

── Q-Learning: V(s) = max_a Q(s,a) ──
[np.float64(1.8), np.float64(3.1), np.float64(4.6), np.float64(6.2)]
[np.float64(2.3), np.float64(0.0), np.float64(6.2), np.float64(8.0)]
[np.float64(4.2), np.float64(6.1), np.float64(0.0), np.float64(10.0)]
[np.float64(1.8), np.float64(8.0), np.float64(10.0), np.float64(0.0)]
